# **Projekt DIGITS**
## 1. Cel: rozpoznanie cyfr 0-9 z nagrań

---
*Autorzy*:
- *Wojciech Salamak*
- *Amelia Sowa*
- *Szymon Stach*
- *Kamil Koterba*



## 2. Plan i architektura systemu

### 1. Architektura i przepływ danych

````mermaid
flowchart LR
    n1["Pliki WAV"] --> n2["Odczyt audio i metadanych z nazwy pliku"]
    n2 --> n3["Przygotowanie sygnału"]
    n3 --> n4["Ekstrakcja cech MFCC"]
    n4 --> n5["Trening"] & n6["Klasyfikacja"]

    n1@{ shape: rect}
    n2@{ shape: rounded}
    n3@{ shape: rounded}
    n4@{ shape: rounded}
    n5@{ shape: lin-proc}
    n6@{ shape: lin-proc}
    style n1 fill:transparent,stroke:#FF6D00
    style n2 fill:transparent
    style n3 fill:transparent
    style n4 fill:transparent
    style n5 fill:transparent,stroke:#00C853
    style n6 fill:transparent,stroke:#2962FF
````

Przyjęto architekturę generatywną. Sygnał audio jest opisywany za pomocą współczynników MFCC. Każda cyfra jest reprezentowana przez osobny model GMM. Metoda ta pozwoli na modelowanie rozkładu cech bez wymagania jednakowej długości nagrań.

System jest podzielony na dwie ścieżki: 
- Trening modeli na nagraniach z etykietami w nazwach plików,
- Klasyfikacja nagrania o nieznanej cyfrze

Obie ścieżki wykorzystują tą samą procedurę przygotowania sygnału i ekstrakcji cech.

````mermaid

flowchart LR
    n6["Powiązanie cech z etykietą"] --> n7["Połączenie ramek z tych samych cyfr"]
    n7 --> n8["Trening osobnego GMM cyfr 0-9"]
    n8 --> n9["Słownik modeli GMM"]
    n5["Trening"] --> n11["Cechy MFCC"]
    n11 --> n6
    n10["Klasyfikacja"] --> n13["Cechy MFCC"]
    n16["Wybór najwyższego wyniku"] --> n17["Rozpoznana cyfra 0-9"]
    n18@{ label: "<span style=\"background-color:\">Obliczanie log-podobieństwa przez każdy model</span><br>" } --> n16
    n13 --> n18
    n15["Słownik modeli GMM"] --> n18

    n6@{ shape: rounded}
    n7@{ shape: rounded}
    n8@{ shape: rounded}
    n9@{ shape: db}
    n5@{ shape: lin-proc}
    n11@{ shape: db}
    n10@{ shape: lin-proc}
    n13@{ shape: db}
    n16@{ shape: rounded}
    n18@{ shape: rounded}
    n15@{ shape: db}
    style n9 stroke:#FFD600
    style n5 fill:transparent,stroke:#00C853
    style n11 stroke:#FF6D00
    style n10 stroke:#2962FF
    style n13 stroke:#FF6D00
    style n17 stroke:#00C853
    style n15 stroke:#FFD600
````

### 2. Struktura danych przechowująca cechy MFCC
Cechy ekstraktowane z nagrań będą przechowywane jako lista. 
Każdy jej element odpowiada jednemu plikowi audio.


```python
recordings = [
    {
        "path": "training_data/JC1M1_4_.wav",
        "speaker": "JC",
        "digit": 4,
        "features": mfcc_matrix
    }
]
```
Pole `features` zawiera macierz Numpy o wymiarze: (liczba_wspolczynników_MFCC, liczba_ramek).
Każdy wiersz opisuje jedna ramkę czasową. 
<p>Osobna macierz dla każdego nagrania pozwoli przechowywać wypowiedzi rożnej długości.

### 3. Struktura danych przechowująca modele GMM
System wykorzysta dziesięć modeli GMM, jeden dla każdej cyfry. 
Wytrenowane modele będą przechowywane w słowniku `models`

```python
models = {
    0: gmm_0,
    1: gmm_1,
    2: gmm_2,
    # ...
    9: gmm_9,
}
```
Każdy model przechowuje wagi komponentów, ich wektory średnich i macierze kowariancje rozkładu cech danej cyfry. 
Słownik ten zwiąże modele z etykietami klas i umożliwi poźniejszą ocenę nagrania.


## 3. Przygotwanie danych

In [50]:
import numpy as np
from sklearn import mixture
from pathlib import Path
import gdown
from pathlib import Path
import re
import librosa
import pickle

In [1]:
### S ### Proponuję napisane funkcje dać do tego bloku, ale jestem otwarty na inne opcje

def filter_recs(recordings:list, **kwargs): ### Zwaraca wszystkie słowniki, które mają podaną w parametrze wartość dla danego klucza
    return [
        d for d in recordings 
        if all(d[k] == v for k, v in kwargs.items())
    ]


def download_data(url, data_dir="data"):
    data_dir = Path(data_dir)
    gdown.download_folder(url=url,output=str(data_dir),quiet=True)
    return data_dir


def train_gmm(mfcc, n, cov_type):
    gmm = mixture.GaussianMixture(n_components=n, covariance_type=cov_type,
                              tol=0.0000001, reg_covar=1e-02, max_iter=100,
                              n_init=1, init_params='random',
                              weights_init=None, means_init=None,precisions_init=None,
                              random_state=16, warm_start=False,
                              verbose=0, verbose_interval=1)
    return gmm.fit(mfcc.T)

In [ ]:
### S ### jak narazie to ładuje się to okropnie długo i nie zawsze całe, więc lepiej pobrać zip i go rozpakować w /training_data/
path = download_data("https://drive.google.com/drive/u/1/folders/1CF7Q7dWQSmmrpj92mDmCYXKQg-nz_Lm_", data_dir='training_data')

In [52]:
data_dir = Path("training_data")
wav_paths = sorted(
    path for path in data_dir.rglob("*")
    if path.is_file() and path.suffix.lower() == ".wav"
)

recordings = []

for path in wav_paths: 
    match = re.search(r"_([0-9])_", path.stem)

    recordings.append({
        "path": path,
        "speaker": path.name[:2],
        "digit": int(match.group(1)),
    })

## 4. Parametryzacja

In [ ]:
for rec in recordings:
    y, sr = librosa.load(rec["path"], sr=16000)
    
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    
    rec["features"] = mfcc

with open("sparametryzowane_dane.pkl", "wb") as plik:
  pickle.dump(recordings, plik)
  
  ### S ### tu by się przydało wrzucić to na dysk, żeby nie trzeba było pobierać całego 

## 5. Model klasyfikacyjny


In [ ]:
### S ### jeśli plik plk jest na dysku, to trzeba zmienić na pobranie z dysku
with open("sparametryzowane_dane.pkl", "rb") as plik:
  recordings = pickle.load(plik)
  
models = {}
for i in range(0,10):
    recs = filter_recs(recordings,digit = i)

    conc_mfcc = np.concatenate([d['features'] for d in recs],1)
    
    #print(f"Nagranie {i}: kształt = {conc_mfcc.shape}")
    models[i] = train_gmm(conc_mfcc,4,'diag')


################### szybki test wsowny, można go usunąć
test = filter_recs(recordings,digit = 8,speaker = "JP")

for i in models.keys():
  print(i,models[i].score(test[0]["features"].T)) ### tu trzeba było zrobić translację
###################


0 -63.54993438720703
1 -62.9239616394043
2 -65.90032958984375
3 -63.493080139160156
4 -65.293701171875
5 -60.66532516479492
6 -66.02178955078125
7 -62.178375244140625
8 -55.93756103515625
9 -62.93886184692383


## 6. Walidacja

## 7. Ewaluacja

## 8. Wnioski